In [ ]:

target = Table.read('/home/hmf/work/FA_test/data/random/Nrepet_test/tertiary-targets-0001.fits')
files = glob.glob(f'/home/hmf/work/FA_test/data/random/Nrepet_test/output/iter0*.fits')


fa_list = []
for f in files:
    tab = Table.read(f)
    tab = tab[tab['FA_TYPE'] == 1]  # 只保留成功分配的目标
    fa_list.append(tab['TARGETID', 'TARGET_RA', 'TARGET_DEC'])

# =========================
# 2. 合并
# =========================
fa_all = vstack(fa_list)

print(f"Total assigned targets: {len(np.unique(fa_all['TARGETID']))/len(target)}")
_, unique_idx = np.unique(fa_all['TARGETID'], return_index=True)

fa_unique = fa_all[unique_idx]

print(len(fa_unique), len(fa_all))



# 假设字段名如下（根据你的文件改）
ra_t, dec_t = target['RA'], target['DEC']
ra_f, dec_f = fa_unique['TARGET_RA'], fa_unique['TARGET_DEC']

T_x, T_y = radec2xy(hw, tile_ra, tile_dec, tile_obstime, tile_theta, tile_obsha,
             ra_t, dec_t, use_cs5=True)
A_x, A_y = radec2xy(hw, tile_ra, tile_dec, tile_obstime, tile_theta, tile_obsha,
             ra_f, dec_f, use_cs5=True)


input_ply = '/home/hmf/work/FA_test/data/focalplane_ply/2025-10-27T17:01:02+00:00_module_edges.ply'

completeness,comple_r, points_t, points_a = compute_completeness(T_x, T_y, A_x, A_y, input_ply, re_source=True)

print(completeness, comple_r, np.shape(points_t), np.shape(points_a))




In [ ]:
fig, ax = plt.subplots(figsize=(24, 20))
ax.scatter(points_t[:, 0], points_t[:, 1], s=1, alpha=1, label='All targets')
ax.scatter(points_a[:, 0], points_a[:, 1], s=1, alpha=1, label='Assigned targets')
plot_ply_with_fill(ply_file, ax=ax)

ax.legend()
ax.set_xlabel('X [mm]')
ax.set_ylabel('Y [mm]')
ax.set_title(f'Completeness: {completeness:.2f} ({comple_r} targets)')
plt.show()


In [ ]:

import os
os.environ["DESIMODEL"] = os.path.expanduser("~/work/FA_test/desi")
print(os.environ["DESIMODEL"])
import subprocess
import numpy as np
from astropy.table import Table




# =========================
# 工具函数
# =========================
def run_cmd(cmd):
    print("\n[RUN]", cmd)
    subprocess.run(cmd, shell=True, check=True)


def run_fba(target_file, prefix):
    cmd = (
        f"fba_run "
        f"--targets {target_file} "
        f"--rundate 2025-10-27T17:01:02+00:00 "
        f"--footprint {footprint} "
        f"--dir {output_dir} "
        f"--overwrite "
        f"--sky_per_module 1 "
        f"--standards_per_module 1 "
        f"--prefix {prefix}"
    )
    run_cmd(cmd)


def find_fba_file(prefix):
    """
    自动找到 fba 输出文件（因为 tileid 可能不止一个）
    """
    files = os.listdir(output_dir)
    matches = [f for f in files if f.startswith(prefix) and f.endswith(".fits")]
    
    if len(matches) == 0:
        raise FileNotFoundError(f"No fba output for prefix {prefix}")
    
    # 如果只有一个 tile，直接用第一个
    return os.path.join(output_dir, matches[0])


def get_assigned_ids(fba_file):
    tab = Table.read(fba_file)

    # 关键：FA_TYPE > 0 表示分配成功
    mask = tab["FA_TYPE"] > 0

    assigned_ids = np.unique(tab["TARGETID"][mask])

    print(f"  Assigned this round: {len(assigned_ids)}")
    return assigned_ids


def update_target(old_target, assigned_ids, new_target):
    t = Table.read(old_target)

    mask = np.isin(t["TARGETID"], assigned_ids)

    # print(f"  Update PRIORITY=0 for {mask.sum()} targets")

    # t["PRIORITY"][mask] = 0
    t = t[~mask]  # 直接删除已分配的目标
    print(f"  Remaining targets for next round: {len(t)}")   

    t.write(new_target, overwrite=True)


# =========================
# 主循环
# =========================

# =========================
# 配置（按你的实际路径）
# =========================
target_init = "/home/hmf/work/FA_test/data/random/Nrepet_test/tertiary-targets-0001.fits"
output_dir = "/home/hmf/work/FA_test/data/random/Nrepet_test/output/"
footprint = "/home/hmf/work/FA_test/data/random/Nrepet_test/tertiary-tiles-0001.fits"

n_iter = 10  # 重复次数

os.makedirs(output_dir, exist_ok=True)

current_target = target_init

for i in range(n_iter):
    print(f"\n========== Iteration {i} ==========")

    prefix = f"iter{i:02d}"

    # 1. 跑 fba
    run_fba(current_target, prefix)

    # 2. 找输出文件
    fba_file = find_fba_file(prefix)
    print(f"  Using fba file: {fba_file}")

    # 3. 读取分配结果
    assigned_ids = get_assigned_ids(fba_file)

    # 4. 生成新 target 文件
    new_target = f"/home/hmf/work/FA_test/data/random/Nrepet_test/tertiary-targets-0001_a{i+1}.fits"
    update_target(current_target, assigned_ids, new_target)

    # 5. 更新
    current_target = new_target